In [113]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re
from nltk.corpus import stopwords
import nltk 
nltk.download("stopwords")

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)


[nltk_data] Downloading package stopwords to C:\Users\ASUS
[nltk_data]     ZENBOOK\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Import Data

In [114]:
data_path = f"../data/Restaurant_ABSA.xlsx"
df = pd.read_excel(data_path)
df.head()

,Id,Comment,"{Aspect category, Sentiment Polarity}"
0,1,খাবারটি অত্যন্ত সুস্বাদু ছিল কিন্তু দাম বেশি ছিল,"{food, positive}, {price, negative}"
1,2,যদিও খাদ্য ভাল ছিল কিন্তু পরিবেষনা ছিল বাজে।।,"{food, positive}, {service, negative}"
2,3,খাদ্যে সবসময় তাজা স্বাদ এবং তাত্ক্ষণিকভাবে পরিবেশিত।,"{food, positive}, {service, positive}"
3,4,খুব ভাল দাম এবং খুব ভাল সেবা।,"{price, positive}, {service, positive}"
4,5,"এটি একটি চমত্কার শিথিল জায়গা, এখনের খাদ্য ভাল ।","{ambiance, positive}, {food, positive}"


## Translate reviews to English

In [115]:
from deep_translator import GoogleTranslator
def translate_text(text):
    try:
        return GoogleTranslator(source='auto', target='en').translate(text)
    except:
        return text

In [116]:
new_data_path = re.sub("xlsx", "csv", data_path)
new_data_path

'../data/Restaurant_ABSA.csv'

In [117]:
if(re.sub("../data/", "", new_data_path) not in os.listdir("../data")):
    df['review_en'] = df['Comment'].apply(translate_text)

    df.to_csv(new_data_path, index=False)
    print("Saved: ", new_data_path)
else:
    df = pd.read_csv(new_data_path)

## Preprocessing

In [118]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 801 entries, 0 to 800
Data columns (total 4 columns):
 #   Column                                   Non-Null Count  Dtype
---  ------                                   --------------  -----
 0   Id                                       801 non-null    int64
 1   Comment                                  801 non-null    str  
 2   {Aspect category,   Sentiment Polarity}  801 non-null    str  
 3   review_en                                801 non-null    str  
dtypes: int64(1), str(3)
memory usage: 252.0 KB


- Rename column for easier retriveval

In [119]:
df = df.rename(columns={"{Aspect category,   Sentiment Polarity}" : "sentiment_detail", "Comment" : "review"})
df = df[["review", "review_en", "sentiment_detail"]]

- Our goal is predicting aspects so we'll just parse aspect list from each record

In [120]:
def parse_aspect(text):
    if not isinstance(text, str):
        print(text)
        return []
    asp_sent = re.findall(r"\{([^,]+),\s*(positive|negative|neutral)\}", text)
    asp = []
    for pair in asp_sent:
        asp.append(pair[0])
    return asp
df['aspect_list'] = df['sentiment_detail'].apply(parse_aspect)

In [121]:
df.head()

,review,review_en,sentiment_detail,aspect_list
0,খাবারটি অত্যন্ত সুস্বাদু ছিল কিন্তু দাম বেশি ছিল,The food was very tasty but the price was high,"{food, positive}, {price, negative}","[food, price]"
1,যদিও খাদ্য ভাল ছিল কিন্তু পরিবেষনা ছিল বাজে।।,Although the food was good but the ambience was bad.,"{food, positive}, {service, negative}","[food, service]"
2,খাদ্যে সবসময় তাজা স্বাদ এবং তাত্ক্ষণিকভাবে পরিবেশিত।,Food always tastes fresh and served promptly.,"{food, positive}, {service, positive}","[food, service]"
3,খুব ভাল দাম এবং খুব ভাল সেবা।,Very good price and very good service.,"{price, positive}, {service, positive}","[price, service]"
4,"এটি একটি চমত্কার শিথিল জায়গা, এখনের খাদ্য ভাল ।","It's a pretty relaxing place, now the food is good.","{ambiance, positive}, {food, positive}","[ambiance, food]"


- Explode dataframe by sentiment detail and split into 2 columns: aspect and sentiment

In [122]:
df = df[df['aspect_list'].notna()]

- Aspect unique values

In [123]:
aspect_list = []
for l in df['aspect_list']:
    for asp in l:
        if asp not in aspect_list:
            aspect_list.append(asp)
aspect_list

['food', 'price', 'service', 'ambiance', 'miscellaneous', 'serice']

Notice that there are a typo error in aspect unique values, so that we'll clean it

In [124]:
for i in range(len(df)):
    new = []
    
    for asp in df.loc[i, 'aspect_list']:
        if asp == 'serice':
            new.append('service')
        else:
            new.append(asp)
    
    df.at[i, 'aspect_list'] = new

aspect_list = []
for l in df['aspect_list']:
    for asp in l:
        if asp not in aspect_list:
            aspect_list.append(asp)

aspect_list

['food', 'price', 'service', 'ambiance', 'miscellaneous']

Check for sample with no aspect

In [125]:
df["num_aspects"] = df["aspect_list"].apply(len)
df[df["num_aspects"] == 0]

,review,review_en,sentiment_detail,aspect_list,num_aspects


Create one-hot label for each aspect

In [126]:
for asp in aspect_list:
    df[asp] = df['aspect_list'].apply(lambda x: 1 if asp in x else 0)

Drop the column that won't be useful for the next session

In [127]:
cols = ['review_en', 'food', 'price', 'service', 'ambiance', 'miscellaneous', 'num_aspects']
df = df[cols]

Check for null data

In [128]:
df.isna().sum()

review_en        0
food             0
price            0
service          0
ambiance         0
miscellaneous    0
num_aspects      0
dtype: int64

After preprocessing for label

In [129]:
df.head()

,review_en,food,price,service,ambiance,miscellaneous,num_aspects
0,The food was very tasty but the price was high,1,1,0,0,0,2
1,Although the food was good but the ambience was bad.,1,0,1,0,0,2
2,Food always tastes fresh and served promptly.,1,0,1,0,0,2
3,Very good price and very good service.,0,1,1,0,0,2
4,"It's a pretty relaxing place, now the food is good.",1,0,0,1,0,2


Next, we'll clean the reviews with the following step:
  - lower text
  - remove special characters
  - remove extra spaces
  - remove stopwords
  - lemmaitization

In [130]:
# Preload stop words to avoid reloading them every time the function is called
stop_words = set(stopwords.words('english'))


def remove_stopwords(text):
    # Removing stop words from the text
    filtered_words = [word for word in text.split() if word not in stop_words]
    cleaned_text = ' '.join(filtered_words)

    return cleaned_text

In [131]:
import contractions
def expand_contractions(text):
    return contractions.fix(text)

In [132]:
nltk.download('wordnet', quiet=True)

def lemmitization(text):
    """Lemmitize text"""
    lemmatizer = nltk.stem.WordNetLemmatizer()
    return ' '.join([lemmatizer.lemmatize(word) for word in text.split()])

In [133]:
import re

def clean_text(text):
    text = expand_contractions(text)
    text = text.lower()
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
    text = text.replace("\u200b", "")
    text = re.sub(r"\s+", " ", text).strip()
    return text

def clean_for_ml(text):
    text = remove_stopwords(text)
    text = lemmitization(text)
    return text

df['review_en'] = df['review_en'].apply(clean_text)
df['review_cleaned'] = df['review_en'].apply(clean_for_ml)

After text preprocessing

In [134]:
df.sample(5)

,review_en,food,price,service,ambiance,miscellaneous,num_aspects,review_cleaned
166,the lemon sorbet of this restaurant is very refreshing and the price is also very low,1,1,0,0,0,2,lemon sorbet restaurant refreshing price also low
765,the food in this restaurant is average but the staff is very good at providing great service,1,0,1,0,0,2,food restaurant average staff good providing great service
690,prices are above average but taste is below average,1,1,0,0,0,2,price average taste average
398,i like everything very much but i will deduct a number because the seat is not very comfortable,0,0,0,1,1,2,like everything much deduct number seat comfortable
108,the quality of kacchi is good but they took a lot of time in delivery,1,0,1,0,0,2,quality kacchi good took lot time delivery


In [135]:
df[df.duplicated()]

,review_en,food,price,service,ambiance,miscellaneous,num_aspects,review_cleaned
47,the food was very tasty but the price was high,1,1,0,0,0,2,food tasty price high
664,the side dishes are not very good but the service is good,1,0,1,0,0,2,side dish good service good
718,the steak was very good but the price was too high,1,1,0,0,0,2,steak good price high
793,customers are happy with the food quality and also happy with the price,1,1,0,0,0,2,customer happy food quality also happy price
800,the quality is good but the quantity is very low compared to the price,1,1,0,0,0,2,quality good quantity low compared price


In [136]:
df = df.drop_duplicates()

In [137]:
df = df[["review_en", "review_cleaned", "food", "price", "service", "ambiance", "miscellaneous"]]

In [138]:
df.info()

<class 'pandas.DataFrame'>
Index: 796 entries, 0 to 799
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   review_en       796 non-null    str  
 1   review_cleaned  796 non-null    str  
 2   food            796 non-null    int64
 3   price           796 non-null    int64
 4   service         796 non-null    int64
 5   ambiance        796 non-null    int64
 6   miscellaneous   796 non-null    int64
dtypes: int64(5), str(2)
memory usage: 138.4 KB


In [139]:
df[["review_en", "food", "price", "service", "ambiance", "miscellaneous"]].head(5)

,review_en,food,price,service,ambiance,miscellaneous
0,the food was very tasty but the price was high,1,1,0,0,0
1,although the food was good but the ambience was bad,1,0,1,0,0
2,food always tastes fresh and served promptly,1,0,1,0,0
3,very good price and very good service,0,1,1,0,0
4,it is a pretty relaxing place now the food is good,1,0,0,1,0


In [140]:
df.to_csv("../Data/Restaurant_ABSA_processed.csv", index=False)